In [ ]:
# Geneva Cycle & Pedestrian Network Analysis
# -------------------------------------------------
# This script downloads OpenStreetMap data for the Canton of Geneva, then
# computes the share (in linear metres and estimated surface) of bicycle lanes
# and pedestrian‑dedicated paths relative to the overall road network. Finally,
# it renders two minimalist maps à la Action Située.
# -------------------------------------------------
# Usage (from a notebook or terminal):
#   %run geneva_cyclability_analysis.py
# or simply execute the cells step by step if you paste it in a notebook.
# -------------------------------------------------
import osmnx as ox
import geopandas as gpd
import pandas as pd
import folium
from shapely.geometry import LineString
from shapely.ops import unary_union
import networkx as nx
import branca.colormap as cm

In [ ]:
# Geneva Cycle & Pedestrian Network Analysis – Faisceaux transfrontaliers
# -------------------------------------------------
# Ce script extrait les infrastructures cyclables et piétonnes le long de deux
# faisceaux transfrontaliers (St-Genis ↔ Genève et St-Julien ↔ Carouge), puis
# calcule les métriques de linéarité et de surface, et produit une carte interactive.
# -------------------------------------------------
import osmnx as ox
import geopandas as gpd
import pandas as pd
import folium
from shapely.geometry import LineString
from shapely.ops import unary_union
import branca.colormap as cm

# -------------------------------------------------
# 1. Définir les faisceaux et les transformer en cercles
# -------------------------------------------------
print("Création des zones d'analyse autour des faisceaux…")
lines = [
    LineString([(6.020712, 46.243278), (6.1422, 46.2102)]),
    LineString([(6.0815, 46.1365), (6.1397, 46.1811)])
]
gdf_lines = gpd.GeoDataFrame(geometry=lines, crs="EPSG:4326").to_crs("EPSG:2056")

buffer_points = []
for line in gdf_lines.geometry:
    for dist in range(0, int(line.length), 1000):
        pt = line.interpolate(dist)
        buffer_points.append(pt.buffer(500))

area_gdf = gpd.GeoDataFrame(geometry=buffer_points, crs="EPSG:2056")
area_wgs = area_gdf.to_crs("EPSG:4326")

# -------------------------------------------------
# 2. Télécharger les graphes OSM pour chaque polygone
# -------------------------------------------------
print("Téléchargement des données OSM pour les faisceaux…")
graphs = []
for geom in area_wgs.geometry:
    try:
        g = ox.graph_from_polygon(geom, network_type="all", simplify=True)
        graphs.append(g)
    except Exception as e:
        print(f"Erreur sur un polygone : {e}")



# Fusionner les graphes via networkx.compose_all
G_all = nx.compose_all(graphs)
edges_all = ox.graph_to_gdfs(G_all, nodes=False, edges=True, fill_edge_geometry=True)
edges_all = edges_all.to_crs("EPSG:2056")

# -------------------------------------------------
# 3. Catégorisation et largeur estimée
# -------------------------------------------------
cycle_highways = {"cycleway"}
ped_highways = {"footway", "pedestrian", "path", "steps"}

if "cycleway" not in edges_all.columns:
    edges_all["cycleway"] = pd.NA
if "width" not in edges_all.columns:
    edges_all["width"] = pd.NA

def estimate_width(row):
    w = row.get("width")
    if isinstance(w, list): w = w[0]
    try:
        return float(str(w).split(";")[0])
    except:
        pass
    hw = row.get("highway")
    if isinstance(hw, list): hw = hw[0]
    if hw in cycle_highways or pd.notnull(row.get("cycleway")):
        return 1.5
    if hw in ped_highways:
        return 2.0
    return 3.5

edges_all["width_est"] = edges_all.apply(estimate_width, axis=1)
edges_all["surface"] = edges_all["length"] * edges_all["width_est"]

# -------------------------------------------------
# 4. Création des indices séparés vélo / marche
# -------------------------------------------------
edges_all["surface_bike"] = 0.0
edges_all["surface_ped"] = 0.0

bike_cond = edges_all["highway"].isin(cycle_highways) | edges_all["cycleway"].notna()
ped_cond = edges_all["highway"].isin(ped_highways)

edges_all.loc[bike_cond, "surface_bike"] = edges_all.loc[bike_cond, "surface"]
edges_all.loc[ped_cond, "surface_ped"] = edges_all.loc[ped_cond, "surface"]

edges_all["ratio_bike"] = edges_all["surface_bike"] / edges_all["surface"]
edges_all["ratio_ped"] = edges_all["surface_ped"] / edges_all["surface"]

# -------------------------------------------------
# 5. Carte interactive
# -------------------------------------------------
print("Création de la carte interactive…")
edges_all = edges_all.to_crs("EPSG:4326")
center = edges_all.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=12, tiles="cartodbpositron")

colormap_bike = cm.linear.YlGn_09.scale(0, 1)
colormap_ped = cm.linear.Blues_09.scale(0, 1)
colormap_bike.caption = "Proportion de surface vélo"
colormap_ped.caption = "Proportion de surface piétonne"
colormap_bike.add_to(m)
colormap_ped.add_to(m)

folium.GeoJson(
    edges_all,
    name="Cyclabilité",
    style_function=lambda e: {
        "color": colormap_bike(e["properties"].get("ratio_bike", 0) or 0),
        "weight": 2
    }
).add_to(m)

folium.GeoJson(
    edges_all,
    name="Marchabilité",
    style_function=lambda e: {
        "color": colormap_ped(e["properties"].get("ratio_ped", 0) or 0),
        "weight": 2
    }
).add_to(m)

folium.LayerControl().add_to(m)

# Aperçu direct dans Jupyter
m

In [ ]:
# Geneva Cycle & Pedestrian Network Analysis – Faisceaux transfrontaliers
# -------------------------------------------------
# Ce script extrait les routes principales dans deux faisceaux transfrontaliers
# (St-Genis ↔ Genève et St-Julien ↔ Carouge), puis identifie les tronçons avec
# attributs cyclables ou piétons, calcule des indices, et produit une carte interactive.
# -------------------------------------------------
import osmnx as ox
import geopandas as gpd
import pandas as pd
import folium
from shapely.geometry import LineString
from shapely.ops import unary_union
import networkx as nx

# -------------------------------------------------
# 1. Définir les faisceaux et zones d'analyse
# -------------------------------------------------
print("Création des zones d'analyse autour des faisceaux…")
lines = [
    #LineString([(6.103282, 46.220386), (6.1422, 46.2102)]),
    LineString([(6.020712, 46.243278), (6.1422, 46.2102)]),
    LineString([(6.081205, 46.143815), (6.102600, 46.162638), (6.1397, 46.1811)])
]
gdf_lines = gpd.GeoDataFrame(geometry=lines, crs="EPSG:4326").to_crs("EPSG:2056")

buffer_points = []
for line in gdf_lines.geometry:
    for dist in range(0, int(line.length), 1000):
        pt = line.interpolate(dist)
        buffer_points.append(pt.buffer(550))

area_gdf = gpd.GeoDataFrame(geometry=buffer_points, crs="EPSG:2056")
area_wgs = area_gdf.to_crs("EPSG:4326")

# -------------------------------------------------
# 2. Télécharger uniquement le réseau routier
# -------------------------------------------------
print("Téléchargement du réseau routier OSM dans chaque zone…")
graphs = []
for geom in area_wgs.geometry:
    try:
        g = ox.graph_from_polygon(geom, network_type="drive", simplify=True)
        graphs.append(g)
    except Exception as e:
        print(f"Erreur sur un polygone : {e}")

G_all = nx.compose_all(graphs)
edges_all = ox.graph_to_gdfs(G_all, nodes=False, edges=True, fill_edge_geometry=True)
edges_all = edges_all.to_crs("EPSG:2056")

# -------------------------------------------------
# 3. Estimation des surfaces
# -------------------------------------------------
for col in ["cycleway", "sidewalk", "footway", "width"]:
    if col not in edges_all.columns:
        edges_all[col] = pd.NA

def estimate_width(row):
    w = row.get("width")
    if isinstance(w, list): w = w[0]
    try:
        return float(str(w).split(";")[0])
    except:
        return 3.5

edges_all["width_est"] = edges_all.apply(estimate_width, axis=1)
edges_all["surface"] = edges_all["length"] * edges_all["width_est"]

# -------------------------------------------------
# 4. Détection des aménagements vélo / marche
# -------------------------------------------------
cycle_tags = ["lane", "track", "opposite", "opposite_lane", "shared_lane"]
ped_tags = ["both", "left", "right", "yes"]

edges_all["surface_bike"] = edges_all.apply(
    lambda r: r.surface if str(r.get("cycleway")) in cycle_tags else 0,
    axis=1
)
edges_all["surface_ped"] = edges_all.apply(
    lambda r: r.surface if str(r.get("sidewalk")) in ped_tags or pd.notna(r.get("footway")) else 0,
    axis=1
)

edges_all["ratio_bike"] = edges_all["surface_bike"] / edges_all["surface"]
edges_all["ratio_ped"] = edges_all["surface_ped"] / edges_all["surface"]

# -------------------------------------------------
# 5. Carte interactive – sans échelle de couleur
# -------------------------------------------------
print("Création de la carte interactive…")
edges_all = edges_all.to_crs("EPSG:4326")
center = edges_all.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=12, tiles="cartodbpositron")

folium.GeoJson(
    edges_all[edges_all["surface_bike"] > 0],
    name="Cyclabilité",
    style_function=lambda e: {"color": "#66c2a5", "weight": 2}
).add_to(m)

folium.GeoJson(
    edges_all[edges_all["surface_ped"] > 0],
    name="Marchabilité",
    style_function=lambda e: {"color": "#8da0cb", "weight": 2}
).add_to(m)

folium.GeoJson(
    edges_all[(edges_all["surface_bike"] == 0) & (edges_all["surface_ped"] == 0)],
    name="Auto",
    style_function=lambda e: {"color": "#cb978d", "weight": 1}
).add_to(m)

folium.LayerControl().add_to(m)

# Aperçu direct dans Jupyter
m


In [ ]:
print(edges_all["sidewalk"].value_counts(dropna=False))
print(edges_all["cycleway"].value_counts(dropna=False))
print(edges_all["highway"].value_counts())
